# Stage 4 — Phase 2.4/2.5/2.6: Model Setup, Prompting, Parsing, Budget Gate

Builds the eval pipeline for one candidate at a time. Starting with **Qwen3-VL-8B-Instruct**
to prove the pipeline end-to-end, then the same structure repeats for InternVL3-8B and
Molmo-7B-D-0924.

**Candidates (locked per decision 2026-07-09):**
- Qwen3-VL: `Qwen/Qwen3-VL-8B-Instruct`
- InternVL3: `OpenGVLab/InternVL3-8B-hf`
- Molmo: `allenai/Molmo-7B-D-0924`

## 0. Configuration + mount

In [ ]:
DRIVE_ROOT   = "/content/drive/MyDrive/pid_project/data"
KAGGLE_DIR   = "kaggle_pid_symbols"
GUPTA_DIR    = "gupta_pid"

from google.colab import drive
drive.mount('/content/drive')

import os, json, time
from pathlib import Path
from PIL import Image

ROOT = Path(DRIVE_ROOT)
kaggle_p = ROOT / KAGGLE_DIR
gupta_p = ROOT / GUPTA_DIR
print("Kaggle:", kaggle_p.exists(), "| Gupta:", gupta_p.exists())

## Common scoring schema (2.4 — the fixed target format)

Every model's raw output gets parsed into this schema, regardless of what format the model
natively emits (bbox-JSON, points, its own dialect). This is deliberately simpler than the
real agent's `DetectionRecord` (`Agent_Pipeline_Facts.md` §2) — we don't need `provenance`
nesting, tag parsing, etc. for benchmarking, just enough to score detection + typing:

```python
{
    "bbox": [x0, y0, x1, y1],   # tile-local pixel coords, xyxy, absolute (not normalized)
    "confidence": float,         # 0.0-1.0
    "entity_type": str | None,   # typing eval (Kaggle) only; None/ignored for Gupta detection eval
}
```

Ground truth (ours, already built) uses the same `bbox` convention — see `parse_yolo_line`/
`yolo_line_to_xyxy` in the Phase 0-1 and Phase 2 notebooks.

## Load Qwen3-VL-8B-Instruct

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

QWEN_MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"

t0 = time.time()
qwen_processor = AutoProcessor.from_pretrained(QWEN_MODEL_ID)
qwen_model = AutoModelForImageTextToText.from_pretrained(
    QWEN_MODEL_ID, torch_dtype=torch.bfloat16, device_map="cuda"
)
print(f"Loaded {QWEN_MODEL_ID} in {time.time()-t0:.1f}s")
print("VRAM used:", f"{torch.cuda.memory_allocated()/1e9:.1f} GB")

## Prompt engineering (2.5) — Qwen3-VL

Ask for strict JSON matching the common schema directly, to minimize parsing work. Qwen3-VL
is documented to be comfortable with bbox-JSON output.

In [ ]:
SYMBOL_DETECTION_PROMPT = """You are analyzing a tile cropped from a Piping & Instrumentation \
Diagram (P&ID). Detect every symbol (valve, instrument, flange, nozzle, safety device, or \
other P&ID symbol) visible in this image.

Respond with ONLY a JSON array, no other text. Each element must have exactly these fields:
{"bbox": [x0, y0, x1, y1], "confidence": <float 0.0-1.0>, "entity_type": "<short type name>"}

Coordinates are absolute pixel coordinates in this image (top-left origin, x right, y down),
NOT normalized. If you see no symbols, respond with an empty array: []"""

def run_qwen(image: Image.Image, prompt: str = SYMBOL_DETECTION_PROMPT):
    messages = [{
        "role": "user",
        "content": [{"type": "image", "image": image}, {"type": "text", "text": prompt}],
    }]
    inputs = qwen_processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors="pt",
    ).to(qwen_model.device)
    t0 = time.time()
    with torch.no_grad():
        out = qwen_model.generate(**inputs, max_new_tokens=2048, do_sample=False)
    latency = time.time() - t0
    gen_tokens = out[:, inputs["input_ids"].shape[1]:]
    text = qwen_processor.batch_decode(gen_tokens, skip_special_tokens=True)[0]
    return text, latency

## Parser (2.5) — round-trip Qwen's raw output into the common schema

In [ ]:
import re

def parse_qwen_output(text):
    """Returns (detections, error). detections is a list of common-schema dicts, or None if
    parsing failed. error is None on success, else a short description."""
    cleaned = text.strip()
    # strip markdown code fences if the model wraps its JSON despite instructions
    cleaned = re.sub(r'^```(?:json)?\s*', '', cleaned)
    cleaned = re.sub(r'\s*```$', '', cleaned)
    try:
        data = json.loads(cleaned)
    except json.JSONDecodeError as e:
        return None, f"JSONDecodeError: {e}"
    if not isinstance(data, list):
        return None, f"expected a JSON array, got {type(data).__name__}"
    detections = []
    for i, item in enumerate(data):
        if not isinstance(item, dict):
            return None, f"item {i} is not an object"
        missing = {"bbox", "confidence", "entity_type"} - item.keys()
        if missing:
            return None, f"item {i} missing fields: {missing}"
        bbox = item["bbox"]
        if not (isinstance(bbox, list) and len(bbox) == 4):
            return None, f"item {i} bbox malformed: {bbox}"
        detections.append({
            "bbox": [float(v) for v in bbox],
            "confidence": float(item["confidence"]),
            "entity_type": str(item["entity_type"]),
        })
    return detections, None

## Round-trip test on one sample tile (2.4 confirm)

In [ ]:
sample_img_path = next((kaggle_p / "images").glob("*.jpg"))
sample_img = Image.open(sample_img_path).convert("RGB")
print("Sample image:", sample_img_path.name, sample_img.size)

raw_text, latency = run_qwen(sample_img)
print(f"Latency: {latency:.2f}s")
print(f"\nRaw output:\n{raw_text[:1000]}")

detections, error = parse_qwen_output(raw_text)
if error:
    print(f"\n❌ PARSE FAILED: {error}")
else:
    print(f"\n✓ Parsed {len(detections)} detections into common schema")
    for d in detections[:5]:
        print(" ", d)